# Healthcare Challenge 3 - Baseline Submission

This notebook provides a simple baseline for **Healthcare Challenge 3: Discharge Readiness Prediction**.

**Goal**: Predict `discharge_ready_day11` (0/1) for each hospital stay
**Metric**: Macro-F1 Score - Higher is better

## Instructions:
1. **Replace API credentials** in the first cell with your team's API key and name
2. **Run all cells** to generate and submit baseline predictions
3. **Check the output** for your submission score

This baseline uses only tabular stay data with a simple Random Forest classifier.


In [ ]:
# 1. Initialize Client and Load Data

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from agentds import BenchmarkClient

# 🔑 REPLACE WITH YOUR CREDENTIALS
client = BenchmarkClient(
    api_key="your api key",        # Get from your team dashboard
    team_name="your team name"     # Your exact team name
)

📂 Loading Healthcare Challenge 3 data...
✅ Data loaded:
   Train stays: (1000, 5)
   Test stays: (1000, 4)
   Train columns: ['stay_id', 'patient_id', 'unit_type', 'admission_reason', 'discharge_ready_day11']
   Test columns: ['stay_id', 'patient_id', 'unit_type', 'admission_reason']


# BEST FINAL CODE
- Macro-F1: 0.8408 \
Choose the thr0688 to submit for best score
  - clai_ch3_v2_submission_cb3seed_exp3_succ_audit1_20260302T064650Z_thr0578.csv -- 0.8352
  - clai_ch3_v2_submission_cb3seed_exp3_succ_audit1_20260302T064650Z_thr0688.csv -- 0.8408 

In [ ]:
import os, json, re, time, random, hashlib, warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix

from catboost import CatBoostClassifier, Pool

warnings.filterwarnings("ignore")

# ==========================================================
# 0) EXPERIMENT META (edit ONLY this block per iteration)
#    -> reversible / auditable / attributable
# ==========================================================
EXPERIMENT_META = {
    "change_points": [
        "Successor baseline: keep EXP3 feature pipeline + CatBoost 3-seed bagging",
        "Process hardening: versioned feature cache + run-tagged outputs + richer iteration jsonl fields",
        "Guardrail: MAX_SUBMISSIONS=2 (OOF best + 1 empirical anchor)",
        "Robustness: auto-fallback log paths if upstream jsonl is read-only",
    ],
    "hypothesis": "Improve auditability / reproducibility without changing core modeling choices.",
    "risk_notes": [
        "Main risk is only file naming / cache behavior differences.",
        "If you need exact old file names, set USE_LEGACY_FILENAMES=True.",
        "If you want to lock exact EXP2 feature cache, keep FEAT_CACHE_IN pointing to vitals_features_cb2seed_exp2.csv.",
    ],
    "fallback_plan": "If anything looks off, set USE_LEGACY_FILENAMES=True and/or FORCE_RETRAIN=1.",
}

# ==========================================================
# 1) CONFIG (stable defaults; only tweak when needed)
# ==========================================================
TEAM, TASK, VERSION = "clai", "ch3", "v2"

BASE_DIR = os.environ.get("CLAI_BASE_DIR", "/mnt/data")
if not os.path.exists(BASE_DIR):
    BASE_DIR = os.getcwd()

BRANCH = os.environ.get("CLAI_BRANCH", "cb3seed_exp3_succ_audit1")
SEEDS = [42, 43, 44]

CV_N_SPLITS = 5
CV_SHUFFLE = True
CV_RANDOM_STATE = 42

FORCE_RETRAIN = bool(int(os.environ.get("FORCE_RETRAIN", "0")))
FEATURE_CACHE_MODE = os.environ.get("FEATURE_CACHE_MODE", "versioned")  # "versioned" | "legacy_single_file"
USE_LEGACY_FILENAMES = bool(int(os.environ.get("USE_LEGACY_FILENAMES", "0")))

# Optional: prefer exact EXP2 cache if present (keeps feature set identical across seeds)
ART_DIR = os.path.join(BASE_DIR, "artifacts", f"{TEAM}_{TASK}_{VERSION}")
os.makedirs(ART_DIR, exist_ok=True)
FEAT_CACHE_IN = os.environ.get("FEAT_CACHE_IN", os.path.join(ART_DIR, "vitals_features_cb2seed_exp2.csv"))

# Submission guardrail
MAX_SUBMISSIONS = int(os.environ.get("MAX_SUBMISSIONS", "2"))
EXTRA_SUBMISSION_THR = [0.688]  # empirical anchor (validated multiple rounds)

# Model params (keep identical to EXP3 unless explicitly testing)
CB_PARAMS = dict(
    loss_function="Logloss",
    depth=4,
    iterations=200,
    learning_rate=0.1,
    l2_leaf_reg=8.0,
    verbose=False,
    allow_writing_files=False,
    thread_count=1
)

# Determinism
PY_SEED = 42
random.seed(PY_SEED)
np.random.seed(PY_SEED)

RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

CKPT_DIR = os.path.join(BASE_DIR, "checkpoints", f"{TEAM}_{TASK}_{VERSION}", f"{RUN_TAG}_{BRANCH}")
os.makedirs(CKPT_DIR, exist_ok=True)

ITER_LOG_PATH = os.path.join(BASE_DIR, f"{TEAM}_{TASK}_{VERSION}_iteration_detail.jsonl")
PSTAR_PATH = os.path.join(BASE_DIR, "PSTAR.json")

if USE_LEGACY_FILENAMES:
    OOF_PATH = os.path.join(BASE_DIR, f"oof_{BRANCH}.csv")
    SWEEP_PATH = os.path.join(BASE_DIR, f"threshold_sweep_{BRANCH}_fine.csv")
    SUB_TEMPLATE = os.path.join(BASE_DIR, f"{TEAM}_{TASK}_{VERSION}_submission_{BRANCH}_thr{{tag}}.csv")
else:
    OOF_PATH = os.path.join(BASE_DIR, f"oof_{BRANCH}_{RUN_TAG}.csv")
    SWEEP_PATH = os.path.join(BASE_DIR, f"threshold_sweep_{BRANCH}_{RUN_TAG}_fine.csv")
    SUB_TEMPLATE = os.path.join(BASE_DIR, f"{TEAM}_{TASK}_{VERSION}_submission_{BRANCH}_{RUN_TAG}_thr{{tag}}.csv")

# ==========================================================
# 2) Helpers
# ==========================================================
def file_sha256(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def ensure_appendable_jsonl(path: str, run_tag: str) -> str:
    # If path exists but can't be appended (read-only), fallback to a writable copy.
    if not os.path.exists(path):
        return path
    try:
        with open(path, "a", encoding="utf-8") as _:
            pass
        return path
    except Exception:
        alt = path.replace(".jsonl", f"_append_{run_tag}.jsonl")
        try:
            with open(path, "r", encoding="utf-8") as src, open(alt, "w", encoding="utf-8") as dst:
                dst.write(src.read())
        except Exception:
            with open(alt, "w", encoding="utf-8") as _:
                pass
        return alt

def ensure_writable_json(path: str, run_tag: str) -> str:
    if not os.path.exists(path):
        return path
    try:
        with open(path, "r+", encoding="utf-8") as _:
            pass
        return path
    except Exception:
        return path.replace(".json", f"_write_{run_tag}.json")

def to_float(x):
    try:
        if x is None:
            return np.nan
        return float(x)
    except Exception:
        return np.nan

def safe_word_count(s: str) -> int:
    s = (s or "").strip()
    if not s:
        return 0
    return len(re.findall(r"\b\w+\b", s.lower()))

def lin_slope(xs, ys):
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    msk = np.isfinite(xs) & np.isfinite(ys)
    xs = xs[msk]; ys = ys[msk]
    if len(xs) < 2:
        return 0.0
    x = xs - xs.mean()
    denom = float(np.sum(x ** 2))
    if denom <= 1e-12:
        return 0.0
    return float(np.sum(x * (ys - ys.mean())) / denom)

def summarize_series(days, values, prefix):
    v = np.asarray([to_float(vv) for vv in values], dtype=float)
    d = np.asarray(days, dtype=float)
    out = {}
    out[f"{prefix}_missing_frac"] = float(np.mean(~np.isfinite(v))) if len(v) > 0 else 1.0
    if len(v) == 0 or np.all(~np.isfinite(v)):
        for st in ["mean", "std", "min", "max", "median", "first", "last", "delta", "slope"]:
            out[f"{prefix}_{st}"] = np.nan
        return out

    out[f"{prefix}_mean"]   = float(np.nanmean(v))
    out[f"{prefix}_std"]    = float(np.nanstd(v))
    out[f"{prefix}_min"]    = float(np.nanmin(v))
    out[f"{prefix}_max"]    = float(np.nanmax(v))
    out[f"{prefix}_median"] = float(np.nanmedian(v))

    idx = np.where(np.isfinite(v))[0]
    first = float(v[idx[0]]); last = float(v[idx[-1]])
    out[f"{prefix}_first"] = first
    out[f"{prefix}_last"]  = last
    out[f"{prefix}_delta"] = float(last - first)
    out[f"{prefix}_slope"] = lin_slope(d[idx], v[idx])
    return out

def thr_to_tag(thr: float) -> str:
    code = int(round(float(thr) * 1000))
    return f"{code:04d}"

def read_jsonl(path: str):
    if not os.path.exists(path):
        return []
    out = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                out.append(json.loads(line))
            except Exception:
                pass
    return out

def write_jsonl_append(path: str, obj: dict):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

ITER_LOG_PATH = ensure_appendable_jsonl(ITER_LOG_PATH, RUN_TAG)
PSTAR_PATH = ensure_writable_json(PSTAR_PATH, RUN_TAG)

# ==========================================================
# 3) Load required inputs (NO leakage sources)
# ==========================================================
stays_train_path = os.path.join(BASE_DIR, "stays_train.csv")
stays_test_path  = os.path.join(BASE_DIR, "stays_test.csv")
patients_path    = os.path.join(BASE_DIR, "patients.csv")
vitals_json_path = os.path.join(BASE_DIR, "vitals_timeseries.json")

for p in [stays_train_path, stays_test_path, patients_path, vitals_json_path]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing required file: {p}")

stays_train = pd.read_csv(stays_train_path)
stays_test  = pd.read_csv(stays_test_path)
patients    = pd.read_csv(patients_path)

if "discharge_ready_day11" not in stays_train.columns:
    raise ValueError("stays_train.csv must contain target column 'discharge_ready_day11'")

# ==========================================================
# 4) Feature engineering (Day1..10 only)
# ==========================================================
WINDOWS = {
    "all":   lambda day: 1 <= day <= 10,
    "early3":lambda day: 1 <= day <= 3,
    "last3": lambda day: 8 <= day <= 10
}
VITALS = ["hr", "sbp", "dbp", "temp_c", "rr"]

afebrile_pattern  = r"\bafebrile\b|\bno fever\b|\bwithout fever\b|\bdenies fever\b|\bfever (complaint )?denied\b|\bfever resolved\b|\bafebrile on recheck\b"
fever_neg_pattern = r"\bfebrile\b|\bspiked temp\b|\bspiked temperature\b|\btemp spike\b|\blow[- ]grade temp\b|\btemp elevation\b|\btemp elevated\b|\bfever\b"

kw_patterns = {
    "afebrile": afebrile_pattern,
    "fever_neg": fever_neg_pattern,

    "room_air": r"\broom air\b|\bon ra\b|\bra\b",
    "weaned_o2": r"\bwean(ed|ing)?\b|\btitrate(d|ing)? down\b|\bdown[- ]?titrate(d|ing)?\b",
    "ambulate": r"\bambulat(ed|ing|ion)?\b|\bwalk(ed|ing)?\b|\bmobiliz(ed|ing|ation)?\b|\bout of bed\b|\boob\b",
    "pt_ot": r"\bphysical therapy\b|\boccupational therapy\b|\bpt\b|\bot\b",
    "tolerating_diet": r"\btolerat(ing|ed)\b.*\bdiet\b|\bregular diet\b|\bdiet tolerated\b|\btolerating po\b|\bpo intake\b",
    "voiding": r"\bvoid(ing|ed)?\b|\burinat(ing|ion|ed)?\b|\badequate urine\b",

    "no_issues": r"\bno issues\b|\bno complaints\b|\bwithout complaint\b",
    "vitals_stable": r"\bvitals? (are )?stable\b|\bvs stable\b|\bhemodynamic(ally)? stable\b|\bhd stable\b",
    "labs_ok": r"\blabs? (are )?(ok|stable|normal)\b|\bwnl\b|\bwithin normal limits\b|\bdowntrending\b",
    "sleep": r"\bslept\b|\bsleep(ing)?\b",
    "pain_ctrl": r"\bpain (is )?(well )?control(led)?\b|\bpain managed\b|\bpain improved\b",
    "home_safety": r"\bhome safety\b|\bfall prevention\b|\beducation (provided|done)\b|\bdischarge teaching\b",

    "oxygen": r"\boxygen\b|\bo2\b|\bnasal cannula\b|\bnc\b|\bhigh flow\b|\bhfnc\b|\bnon[- ]?rebreather\b|\bnrb\b|\bbipap\b|\bcpap\b|\bvent\b",
    "resp_worse": r"\bshort(ness)? of breath\b|\bsob\b|\bdesaturat(ed|ion|ing)?\b|\bhypox(ia|ic)\b|\bresp(iratory)? distress\b|\brespiratory failure\b",

    "culture": r"\bculture(s)?\b|\bblood culture\b|\bwound culture\b|\burine culture\b",
    "pending": r"\bpending\b|\bawaiting\b|\bto follow\b",
    "monitor_reassess": r"\bmonitor(ing|ed)?\b|\breassess\b|\bwatch(ing)?\b|\bfollow up\b",
    "broadened": r"\bbroaden(ed|ing)?\b|\bescalat(ed|ing|ion)?\b",
    "sirs": r"\bsirs\b|\bsepsis\b|\bseptic\b",
    "abx": r"\babx\b|\bantibiotic(s)?\b|\bantimicrobial(s)?\b",
    "abx_names": r"\b(vancomycin|meropenem|ertapenem|imipenem|daptomycin|piperacillin|nafcillin|moxifloxacin|azithromycin|amoxicillin|ampicillin|levofloxacin|ciprofloxacin|ceftriaxone|cefepime|zosyn)\b",
    "rehab": r"\brehab\b|\bsnf\b|\bskilled nursing\b|\bplacement\b",
    "bandemia": r"\bbandemia\b|\bbands?\b",
}

RATIO_FUNCS = {
    "shock": lambda hr, sbp, dbp, temp, rr: (hr / sbp) if (np.isfinite(hr) and np.isfinite(sbp) and sbp != 0) else np.nan,
    "pulse_pressure": lambda hr, sbp, dbp, temp, rr: (sbp - dbp) if (np.isfinite(sbp) and np.isfinite(dbp)) else np.nan,
    "map": lambda hr, sbp, dbp, temp, rr: ((2 * dbp + sbp) / 3.0) if (np.isfinite(sbp) and np.isfinite(dbp)) else np.nan,
    "rr_hr_ratio": lambda hr, sbp, dbp, temp, rr: (rr / hr) if (np.isfinite(rr) and np.isfinite(hr) and hr != 0) else np.nan,
}

# Feature config hash -> cache key (prevents "silent old cache" bugs)
FEATURE_CONFIG_FOR_HASH = {
    "windows": ["all(1-10)", "early3(1-3)", "last3(8-10)"],
    "vitals": VITALS,
    "ratios": list(RATIO_FUNCS.keys()),
    "kw_patterns": kw_patterns,
    "abnormal_thresholds": {"fever_temp_c_ge": 38.0, "tachy_hr_ge": 100.0, "hypot_sbp_le": 90.0, "tachyp_rr_ge": 22.0},
    "notes_stats": ["note_len", "note_wc", "kw_binary", "kw_daycnt"],
}
_feature_cfg_blob = json.dumps(FEATURE_CONFIG_FOR_HASH, ensure_ascii=True, sort_keys=True).encode("utf-8")
FEATURE_SET_ID = hashlib.sha256(_feature_cfg_blob).hexdigest()[:16]

if FEATURE_CACHE_MODE == "legacy_single_file":
    FEAT_CACHE_OUT = os.path.join(ART_DIR, f"vitals_features_{BRANCH}.csv")
else:
    FEAT_CACHE_OUT = os.path.join(ART_DIR, f"vitals_features_{FEATURE_SET_ID}.csv")

_kw_compiled = {k: re.compile(pat) for k, pat in kw_patterns.items()}

def build_vitals_features(vitals_json_path: str, out_csv_path: str) -> pd.DataFrame:
    with open(vitals_json_path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    rows = []
    for obj in raw:
        sid = obj["stay_id"]
        days = sorted(obj.get("days", []), key=lambda x: x.get("day", 0))
        days = [d for d in days if d.get("day") is not None and 1 <= int(d.get("day")) <= 10]  # Day1..10 only

        day_nums = [int(d.get("day")) for d in days]
        notes = [(d.get("note") or "") for d in days]
        low_notes = [n.lower() for n in notes]

        feats = {"stay_id": sid}

        # day-level keyword hits (for day-count)
        kw_day_hit = {k: [] for k in _kw_compiled}
        for day, txt in zip(day_nums, low_notes):
            for k, creg in _kw_compiled.items():
                kw_day_hit[k].append((day, 1 if creg.search(txt) else 0))

        # note stats + keyword binary/daycnt in each window
        for wname, wfn in WINDOWS.items():
            txt = " ".join([t for t, day in zip(low_notes, day_nums) if wfn(day)]).strip()
            feats[f"note_len_{wname}"] = int(len(txt))
            feats[f"note_wc_{wname}"]  = int(safe_word_count(txt))

            for k, creg in _kw_compiled.items():
                feats[f"kw_{k}_{wname}"] = int(bool(creg.search(txt)))
                feats[f"kw_{k}_{wname}_daycnt"] = int(sum(hit for day, hit in kw_day_hit[k] if wfn(day)))

        # abnormal day counts (per window)
        def get_vals(key):
            return [to_float(d.get(key, np.nan)) for d in days]

        temp_vals = get_vals("temp_c")
        hr_vals   = get_vals("hr")
        sbp_vals  = get_vals("sbp")
        rr_vals   = get_vals("rr")

        def count_cond(vals, cond, wfn):
            c = 0
            for day, val in zip(day_nums, vals):
                if (not wfn(day)) or (not np.isfinite(val)):
                    continue
                if cond(float(val)):
                    c += 1
            return c

        for wname, wfn in WINDOWS.items():
            feats[f"fever_days_{wname}"]  = count_cond(temp_vals, lambda x: x >= 38.0, wfn)
            feats[f"tachy_days_{wname}"]  = count_cond(hr_vals,   lambda x: x >= 100.0, wfn)
            feats[f"hypot_days_{wname}"]  = count_cond(sbp_vals,  lambda x: x <= 90.0, wfn)
            feats[f"tachyp_days_{wname}"] = count_cond(rr_vals,   lambda x: x >= 22.0, wfn)

        # vitals + ratios per day
        vital_arrays = {v: get_vals(v) for v in VITALS}
        ratio_arrays = {name: [] for name in RATIO_FUNCS}
        for d in days:
            hr   = to_float(d.get("hr"))
            sbp  = to_float(d.get("sbp"))
            dbp  = to_float(d.get("dbp"))
            temp = to_float(d.get("temp_c"))
            rr   = to_float(d.get("rr"))
            for name, fn in RATIO_FUNCS.items():
                ratio_arrays[name].append(fn(hr, sbp, dbp, temp, rr))

        # window stats
        for name, arr in list(vital_arrays.items()) + list(ratio_arrays.items()):
            for wname, wfn in WINDOWS.items():
                vals   = [val for val, day in zip(arr, day_nums) if wfn(day)]
                days_w = [day for day in day_nums if wfn(day)]
                feats.update(summarize_series(days_w, vals, f"{name}_{wname}"))

        # stability features
        for name in VITALS + list(RATIO_FUNCS.keys()):
            m_last  = feats.get(f"{name}_last3_mean", np.nan)
            m_early = feats.get(f"{name}_early3_mean", np.nan)
            feats[f"{name}_late_minus_early_mean"] = float(m_last - m_early) if (np.isfinite(m_last) and np.isfinite(m_early)) else np.nan

            std_last = feats.get(f"{name}_last3_std", np.nan)
            std_all  = feats.get(f"{name}_all_std", np.nan)
            feats[f"{name}_std_ratio_last3_all"] = float(std_last / std_all) if (np.isfinite(std_last) and np.isfinite(std_all) and std_all != 0) else np.nan

        rows.append(feats)

    df = pd.DataFrame(rows)
    df.to_csv(out_csv_path, index=False)
    return df

# Prefer exact EXP2 cache (if available) to keep feature pipeline identical across seeds,
# otherwise use versioned cache keyed by FEATURE_SET_ID.
if (not FORCE_RETRAIN) and (FEAT_CACHE_IN is not None) and os.path.exists(FEAT_CACHE_IN):
    vitals_feat = pd.read_csv(FEAT_CACHE_IN)
    feat_source = FEAT_CACHE_IN
    feat_cache_action = "loaded_feat_cache_in"
else:
    if (not FORCE_RETRAIN) and os.path.exists(FEAT_CACHE_OUT):
        vitals_feat = pd.read_csv(FEAT_CACHE_OUT)
        feat_source = FEAT_CACHE_OUT
        feat_cache_action = "loaded_feat_cache_out"
    else:
        print(f"[feature] building from raw vitals_timeseries.json (FORCE_RETRAIN={FORCE_RETRAIN})")
        vitals_feat = build_vitals_features(vitals_json_path, FEAT_CACHE_OUT)
        feat_source = FEAT_CACHE_OUT
        feat_cache_action = "rebuilt_feat_cache_out"

# ==========================================================
# 5) Merge tables + derived cats + freq encoding
# ==========================================================
train_df = stays_train.merge(patients, on="patient_id", how="left").merge(vitals_feat, on="stay_id", how="left")
test_df  = stays_test.merge(patients, on="patient_id", how="left").merge(vitals_feat, on="stay_id", how="left")

y = train_df["discharge_ready_day11"].astype(int).values
train_pos_rate = float(np.mean(y))
pid_overlap = len(set(train_df["patient_id"]) & set(test_df["patient_id"]))

def add_derived_cats(df):
    df = df.copy()
    df["unit_reason"] = df["unit_type"].astype(str).fillna("UNK") + "__" + df["admission_reason"].astype(str).fillna("UNK")
    df["age_bin"] = (np.floor(df["age"].astype(float) / 10) * 10).clip(0, 120).astype("Int64").astype(str)
    return df

train_df = add_derived_cats(train_df)
test_df  = add_derived_cats(test_df)

cat_cols = ["unit_type", "admission_reason", "sex", "insurance", "zip3", "unit_reason", "age_bin"]
for c in cat_cols:
    train_df[c] = train_df[c].astype(str).fillna("UNK").replace({"nan":"UNK","<NA>":"UNK","None":"UNK"})
    test_df[c]  = test_df[c].astype(str).fillna("UNK").replace({"nan":"UNK","<NA>":"UNK","None":"UNK"})

n_train = len(train_df)
for c in cat_cols:
    freq = train_df[c].value_counts(dropna=False) / n_train
    train_df[f"freq_{c}"] = train_df[c].map(freq).astype(float)
    test_df[f"freq_{c}"]  = test_df[c].map(freq).fillna(0).astype(float)

drop_cols = ["stay_id", "patient_id", "discharge_ready_day11"]
feature_cols = [c for c in train_df.columns if c not in drop_cols]
X = train_df[feature_cols]
X_test = test_df[feature_cols]
cat_idx = [feature_cols.index(c) for c in cat_cols]

# ==========================================================
# 6) CV -> OOF (3-seed bagging)
# ==========================================================
skf = StratifiedKFold(n_splits=CV_N_SPLITS, shuffle=CV_SHUFFLE, random_state=CV_RANDOM_STATE)

fold_ids = np.full(len(X), -1, dtype=int)
for fold, (_, va_idx) in enumerate(skf.split(X, y)):
    fold_ids[va_idx] = fold

fold_hash_sha256 = hashlib.sha256(",".join([f"{sid}:{f}" for sid, f in zip(train_df["stay_id"].tolist(), fold_ids.tolist())]).encode()).hexdigest()

t0 = time.time()
oof_by_seed = {}
for seed in SEEDS:
    oof = np.zeros(len(X), dtype=float)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        tr_pool = Pool(X.iloc[tr_idx], y[tr_idx], cat_features=cat_idx)
        va_pool = Pool(X.iloc[va_idx], cat_features=cat_idx)
        model = CatBoostClassifier(**CB_PARAMS, random_seed=seed)
        model.fit(tr_pool)
        oof[va_idx] = model.predict_proba(va_pool)[:, 1]
    oof_by_seed[seed] = oof

oof_avg = np.mean(np.column_stack([oof_by_seed[s] for s in SEEDS]), axis=1)

# ==========================================================
# 7) Threshold sweep (coarse -> fine) + richer sweep CSV
# ==========================================================
thr_coarse = np.arange(0.05, 0.951, 0.01)
coarse_scores = [f1_score(y, (oof_avg >= thr).astype(int), average="macro") for thr in thr_coarse]
coarse_best_thr = float(thr_coarse[int(np.argmax(coarse_scores))])

thr_fine = np.arange(max(0.05, coarse_best_thr - 0.10), min(0.95, coarse_best_thr + 0.10) + 1e-12, 0.001)
rows = []
for thr in thr_fine:
    pred = (oof_avg >= thr).astype(int)
    f1_0, f1_1 = f1_score(y, pred, average=None, labels=[0, 1])
    rows.append((float(thr), float((f1_0 + f1_1) / 2.0), float(f1_0), float(f1_1), float(np.mean(pred))))

sweep_df = pd.DataFrame(rows, columns=["thr", "macro_f1", "f1_class0", "f1_class1", "pos_rate"])
best_row = sweep_df.sort_values("macro_f1", ascending=False).iloc[0]
best_thr = float(best_row["thr"])
best_macro_f1 = float(best_row["macro_f1"])
best_pos_rate = float(best_row["pos_rate"])

per_fold_macro = [float(f1_score(y[fold_ids == f], (oof_avg[fold_ids == f] >= best_thr).astype(int), average="macro")) for f in range(CV_N_SPLITS)]
cm = confusion_matrix(y, (oof_avg >= best_thr).astype(int)).tolist()

anchor_thresholds = [best_thr] + EXTRA_SUBMISSION_THR
anchor_report = {}
for thr in sorted(set([float(x) for x in anchor_thresholds])):
    pred = (oof_avg >= thr).astype(int)
    f1_0, f1_1 = f1_score(y, pred, average=None, labels=[0, 1])
    anchor_report[f"{thr:.3f}"] = {
        "macro_f1": float((f1_0 + f1_1) / 2.0),
        "f1_class0": float(f1_0),
        "f1_class1": float(f1_1),
        "pos_rate": float(np.mean(pred)),
    }

# ==========================================================
# 8) Fit full models -> predict test -> write submissions
# ==========================================================
full_pool = Pool(X, y, cat_features=cat_idx)
test_pool = Pool(X_test, cat_features=cat_idx)

test_pred_by_seed = {}
for seed in SEEDS:
    model = CatBoostClassifier(**CB_PARAMS, random_seed=seed)
    model.fit(full_pool)
    test_pred_by_seed[seed] = model.predict_proba(test_pool)[:, 1]
    model.save_model(os.path.join(CKPT_DIR, f"catboost_full_seed{seed}.cbm"))

test_pred_avg = np.mean(np.column_stack([test_pred_by_seed[s] for s in SEEDS]), axis=1)

def write_submission(thr: float) -> str:
    tag = thr_to_tag(thr)
    out_path = SUB_TEMPLATE.format(tag=tag)
    sub = pd.DataFrame({
        "stay_id": test_df["stay_id"].astype(int).values,
        "discharge_ready_day11": (test_pred_avg >= float(thr)).astype(int)
    })
    sub.to_csv(out_path, index=False)
    return out_path

thr_candidates = [best_thr] + [float(t) for t in EXTRA_SUBMISSION_THR]
seen_tags = set()
thr_final = []
for thr in thr_candidates:
    tag = thr_to_tag(thr)
    if tag in seen_tags:
        continue
    seen_tags.add(tag)
    thr_final.append(float(thr))
    if len(thr_final) >= MAX_SUBMISSIONS:
        break

subs_written = {}
for thr in thr_final:
    tag = thr_to_tag(thr)
    subs_written[f"thr{tag}"] = write_submission(thr)

# ==========================================================
# 9) Write OOF + sweep + hashes
# ==========================================================
oof_out = pd.DataFrame({
    "stay_id": train_df["stay_id"].astype(int).values,
    "y_true": y.astype(int),
    "fold": fold_ids.astype(int),
})
for seed in SEEDS:
    oof_out[f"oof_seed{seed}"] = oof_by_seed[seed]
oof_out["oof_proba"] = oof_avg
oof_out.to_csv(OOF_PATH, index=False)

sweep_df.to_csv(SWEEP_PATH, index=False)

sha = {
    os.path.basename(OOF_PATH): file_sha256(OOF_PATH),
    os.path.basename(SWEEP_PATH): file_sha256(SWEEP_PATH),
}
for _, path in subs_written.items():
    sha[os.path.basename(path)] = file_sha256(path)

input_sha = {
    "stays_train.csv": file_sha256(stays_train_path),
    "stays_test.csv": file_sha256(stays_test_path),
    "patients.csv": file_sha256(patients_path),
    "vitals_timeseries.json": file_sha256(vitals_json_path),
}

# ==========================================================
# 10) Append iteration log (append-only) + update PSTAR.json
# ==========================================================
hist = read_jsonl(ITER_LOG_PATH)
prev_ids = [r.get("iteration_id") for r in hist if isinstance(r.get("iteration_id"), int)]
next_id = (max(prev_ids) + 1) if prev_ids else 1

prev_pstar = {}
if os.path.exists(PSTAR_PATH):
    try:
        with open(PSTAR_PATH, "r", encoding="utf-8") as f:
            prev_pstar = json.load(f)
    except Exception:
        prev_pstar = {}

prev_best_oof = float(prev_pstar.get("best_macro_f1_oof", -1.0))

iter_record = {
    "iteration_id": next_id,
    "timestamp_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "branch": BRANCH,
    "run_tag": RUN_TAG,
    "phase": "Modeling",
    "real_score": None,
    "experiment_meta": EXPERIMENT_META,

    "validation_protocol": {
        "cv": f"StratifiedKFold(n_splits={CV_N_SPLITS}, shuffle={CV_SHUFFLE}, random_state={CV_RANDOM_STATE})",
        "fold_hash_sha256": fold_hash_sha256,
        "train_positive_rate": train_pos_rate,
        "train_test_patient_id_overlap": pid_overlap,
    },
    "bagging": {"seeds": SEEDS, "oof_blend": "mean"},
    "model": {"type": "CatBoostClassifier", "params": CB_PARAMS},

    "feature_summary": {
        "feature_set_id": FEATURE_SET_ID,
        "feat_source": feat_source,
        "feat_cache_action": feat_cache_action,
        "windows": ["all(1-10)", "early3(1-3)", "last3(8-10)"],
        "ratios": list(RATIO_FUNCS.keys()),
        "keywords": {
            "afebrile_pattern": afebrile_pattern,
            "fever_neg_pattern": fever_neg_pattern,
            "other_kw_names": [k for k in kw_patterns.keys() if k not in ["afebrile", "fever_neg"]],
        },
        "cat_cols": cat_cols,
        "freq_encoding": True,
        "n_features_total": int(len(feature_cols)),
    },

    "thresholding": {
        "coarse_step": 0.01,
        "fine_step": 0.001,
        "fine_window": 0.10,
        "coarse_best_thr": coarse_best_thr,
        "best_thr_fine": best_thr,
        "submitted_thresholds": thr_final,
        "max_submissions_guardrail": MAX_SUBMISSIONS,
    },

    "metrics": {
        "best_macro_f1": best_macro_f1,
        "best_pos_rate": best_pos_rate,
        "per_fold_macro_f1_at_best": per_fold_macro,
        "confusion_matrix_at_best": cm,
        "anchor_threshold_report": anchor_report,
        "prev_best_macro_f1_oof": prev_best_oof,
        "delta_vs_prev_best_oof": float(best_macro_f1 - prev_best_oof) if prev_best_oof >= 0 else None,
    },

    "artifacts": {
        "oof": OOF_PATH,
        "threshold_sweep_fine": SWEEP_PATH,
        "submissions": subs_written,
        "checkpoint_dir": CKPT_DIR,
        "iteration_log_used": ITER_LOG_PATH,
        "pstar_used": PSTAR_PATH,
    },

    "sha256": sha,
    "input_sha256": input_sha,

    "env": {
        "python": __import__("sys").version.split()[0],
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "catboost": __import__("catboost").__version__,
        "sklearn": __import__("sklearn").__version__,
    }
}
write_jsonl_append(ITER_LOG_PATH, iter_record)

if best_macro_f1 > prev_best_oof:
    new_pstar = dict(prev_pstar)
    new_pstar.update({
        "best_macro_f1_oof": best_macro_f1,
        "best_thr_oof": best_thr,
        "best_branch": BRANCH,
        "best_iteration_id": next_id,
        "updated_utc": iter_record["timestamp_utc"],
    })
    with open(PSTAR_PATH, "w", encoding="utf-8") as f:
        json.dump(new_pstar, f, ensure_ascii=False, indent=2)

# ==========================================================
# 11) Print summary (audit-friendly)
# ==========================================================
print(f"=== {TEAM.upper()} {TASK.upper()} {VERSION} | {BRANCH} | {RUN_TAG} ===")
print(f"BASE_DIR = {BASE_DIR}")
print(f"FEATURE_SET_ID = {FEATURE_SET_ID} | cache_mode={FEATURE_CACHE_MODE} | feat_cache_action={feat_cache_action}")
print(f"feat_source = {feat_source}")
print(f"SEEDS = {SEEDS} | CV = StratifiedKFold(n_splits={CV_N_SPLITS}, shuffle={CV_SHUFFLE}, random_state={CV_RANDOM_STATE})")
print(f"python={iter_record['env']['python']} numpy={iter_record['env']['numpy']} pandas={iter_record['env']['pandas']}")
print(f"catboost={iter_record['env']['catboost']} sklearn={iter_record['env']['sklearn']}")
print(f"fold_hash_sha256 = {fold_hash_sha256}")
print(f"train_positive_rate = {train_pos_rate:.3f} | train/test patient_id overlap = {pid_overlap}")

print(f"\n[OOF] best_thr (fine) = {best_thr:.3f} | best_macro_f1 = {best_macro_f1:.6f} | pos_rate = {best_pos_rate:.3f}")
print(f"[OOF] per-fold macro_f1 @ best_thr: {[round(x,6) for x in per_fold_macro]}")
print(f"[OOF] confusion_matrix @ best_thr: {cm}")

print("\n[OOF] submitted thresholds (guardrail applied):")
for thr in thr_final:
    rep = anchor_report.get(f"{thr:.3f}", None)
    if rep is None:
        pred = (oof_avg >= thr).astype(int)
        f1_0, f1_1 = f1_score(y, pred, average=None, labels=[0, 1])
        rep = {"macro_f1": float((f1_0 + f1_1) / 2.0), "pos_rate": float(np.mean(pred))}
    print(f"  thr={thr:.3f} -> macro_f1={rep['macro_f1']:.6f} | pos_rate={rep['pos_rate']:.3f}")

print("\nWROTE:")
print(f"  {OOF_PATH}")
print(f"  {SWEEP_PATH}")
for k, path in subs_written.items():
    print(f"  [submission {k}] {path}")
print(f"  checkpoint dir -> {CKPT_DIR}")
print(f"  appended iteration log -> {ITER_LOG_PATH}")
print(f"  updated PSTAR (if improved) -> {PSTAR_PATH}")

print("\n[sha256 outputs]")
for name, h in sha.items():
    print(f"  {name} = {h}")

print("\n[sha256 inputs]")
for name, h in input_sha.items():
    print(f"  {name} = {h}")

print(f"\n(done) elapsed_sec = {time.time() - t0:.1f}")

# Submission

In [3]:
# 3. Submit Predictions

# Submit predictions to the competition
print("🚀 Submitting predictions...")

try:
    result = client.submit_prediction("Healthcare", 3, "healthcare_challenge3_predictions.csv")
    
    if result['success']:
        print("✅ Submission successful!")
        print(f"   📊 Score: {result['score']:.4f}")
        print(f"   📏 Metric: {result['metric_name']}")
        print(f"   ✔️  Validation: {'Passed' if result['validation_passed'] else 'Failed'}")
    else:
        print("❌ Submission failed!")
        print(f"   Error details: {result.get('details', {}).get('validation_errors', 'Unknown error')}")
        
except Exception as e:
    print(f"💥 Submission error: {e}")
    print("🔧 Check your API key and team name are correct!")

print("\n🎯 Next steps:")
print("   1. Try incorporating relevant information outside this table!")
print("   2. You've completed all Healthcare challenges!")


🚀 Submitting predictions...
✅ Prediction submitted successfully!
📊 Score: 0.4889 (Macro-F1)
✅ Validation passed
✅ Submission successful!
   📊 Score: 0.4889
   📏 Metric: Macro-F1
   ✔️  Validation: Passed

🎯 Next steps:
   1. Try incorporating relevant information outside this table!
   2. You've completed all Healthcare challenges!
